# Day 2 — Samples, baselines, first CNN train

Load Day 1 outputs from Drive → build lead-3 training samples → score persistence/climatology → train a small CNN.

**Goal:** ugly but working end-to-end. Loss should drop; you know the persistence number to beat.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aarib-sami/ninonet/blob/main/enso/day2_train.ipynb)

Runtime → Change runtime type → **GPU** (optional but nicer).

## 0. Install

In [ ]:
# torch comes preinstalled on Colab; reinstall only if needed
# Packages below (what each one is for today):
#   xarray       — open Day 1's pacific_anom.nc (labeled lat/lon/time arrays)
#   netCDF4      — low-level reader xarray uses for .nc files
#   numpy        — stack sliding-window samples into (N, 12, lat, lon) arrays
#   pandas       — load oni_monthly.csv and align ONI to SST months
#   scikit-learn — baseline metrics (RMSE via mean_squared_error)
#   matplotlib   — quick plots of loss / predictions later
!pip install -q xarray netCDF4 numpy pandas scikit-learn matplotlib


## 1. Mount Drive and load Day 1 files

In [ ]:
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = Path("/content/drive/MyDrive/ensocast/data")
OUT_DIR = Path("/content/drive/MyDrive/ensocast/artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data:", DATA_DIR)
print("Artifacts:", OUT_DIR)
assert (DATA_DIR / "pacific_anom.nc").exists(), "Missing pacific_anom.nc — rerun Day 1"
assert (DATA_DIR / "oni_monthly.csv").exists(), "Missing oni_monthly.csv — rerun Day 1"

In [ ]:
import numpy as np   # fast numeric arrays for X, y, and baselines
import pandas as pd  # tabular ONI + datetime alignment to SST months
import xarray as xr  # open the netCDF anomaly stack with lat/lon/time coords

anom = xr.open_dataarray(DATA_DIR / "pacific_anom.nc")
# Day 1 saved a DataArray; if it's a Dataset, grab the only data var
if isinstance(anom, xr.Dataset):
    anom = anom[list(anom.data_vars)[0]]

oni_df = pd.read_csv(DATA_DIR / "oni_monthly.csv", parse_dates=["time"])

print(anom)
print(oni_df.head())
print("anom months:", anom.sizes.get("time", len(anom["time"])))
print("oni rows:", len(oni_df))


## 2. Build sliding-window samples (lead = 3)

Each sample:
- **X** = 12 consecutive anomaly maps `(12, lat, lon)`
- **y** = ONI **3 months after** the last input month

We also keep `t_end` = date of the last input month (used for the time split).

In [ ]:
WINDOW = 12
LEAD = 3

arr = anom.values.astype("float32")  # (time, lat, lon)
times = pd.to_datetime(anom["time"].values)
# month-start timestamps so CSV dates match SST coords (Day 1 style)
times = times.to_period("M").to_timestamp()

oni_series = oni_df.set_index("time")["oni"]
oni_series.index = pd.to_datetime(oni_series.index).to_period("M").to_timestamp()
oni = oni_series.reindex(times).to_numpy(dtype="float32")

# Day 1 often saved full SST (incl. newest month) but ONI only through last published season
missing = int(np.isnan(oni).sum())
if missing:
    print(f"Dropping {missing} SST month(s) with no ONI (usually the newest).")
    valid = ~np.isnan(oni)
    arr = arr[valid]
    times = times[valid]
    oni = oni[valid]

assert not np.isnan(oni).any(), "ONI still has gaps vs SST times — re-run Day 1 align"
assert len(arr) == len(oni) == len(times)

# t = index of last input month; predict ONI at t+LEAD
X_list, y_list, t_end_list, oni_at_t_list = [], [], [], []
for t in range(WINDOW - 1, len(arr) - LEAD):
    X_list.append(arr[t - WINDOW + 1 : t + 1])  # 12 maps ending at t
    y_list.append(oni[t + LEAD])
    t_end_list.append(times[t])
    oni_at_t_list.append(oni[t])  # persistence: ONI stays as it is today

X = np.stack(X_list).astype("float32")  # (N, 12, lat, lon)
y = np.array(y_list, dtype="float32")
t_end = pd.to_datetime(t_end_list)
oni_at_t = np.array(oni_at_t_list, dtype="float32")

print("X", X.shape, "y", y.shape)
print("first t_end", t_end[0].date(), "last t_end", t_end[-1].date())


## 3. Time split (never random)

- **train:** last input month ≤ 2005-12
- **val:** 2006-01 … 2015-12
- **test:** 2016-01 onward

In [ ]:
train_mask = t_end <= "2005-12-31"
val_mask = (t_end >= "2006-01-01") & (t_end <= "2015-12-31")
test_mask = t_end >= "2016-01-01"

splits = {
    "train": train_mask,
    "val": val_mask,
    "test": test_mask,
}
for name, mask in splits.items():
    print(f"{name:5s}: n={mask.sum():4d}  {t_end[mask][0].date()} → {t_end[mask][-1].date()}")

## 4. Baselines on the test set

- **Persistence:** predict ONI at t+lead = ONI at last input month  
- **Climatology:** always predict 0 (neutral)

Metrics: RMSE and correlation. These are the goalposts.

In [ ]:
from sklearn.metrics import mean_squared_error


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def corr(y_true, y_pred):
    if np.std(y_true) < 1e-8 or np.std(y_pred) < 1e-8:
        return float("nan")
    return float(np.corrcoef(y_true, y_pred)[0, 1])


y_test = y[test_mask]
pers_test = oni_at_t[test_mask]
clim_test = np.zeros_like(y_test)

baselines = {
    "persistence": (pers_test, rmse(y_test, pers_test), corr(y_test, pers_test)),
    "climatology": (clim_test, rmse(y_test, clim_test), corr(y_test, clim_test)),
}

for name, (_, r, c) in baselines.items():
    print(f"{name:12s}  RMSE={r:.3f}  corr={c:.3f}")

print("\nBeat persistence RMSE to claim skill at lead 3.")

## 5. Small CNN

Short ocean record ⇒ keep the net tiny so it doesn't memorize.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


class ENSOForecaster(nn.Module):
    def __init__(self, in_months=12):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_months, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


model = ENSOForecaster().to(device)
print(model)
print("params:", sum(p.numel() for p in model.parameters()))

## 6. DataLoaders

In [ ]:
def make_loader(mask, batch_size=32, shuffle=False):
    ds = TensorDataset(
        torch.from_numpy(X[mask]),
        torch.from_numpy(y[mask]),
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


train_loader = make_loader(train_mask, shuffle=True)
val_loader = make_loader(val_mask, shuffle=False)
test_loader = make_loader(test_mask, shuffle=False)

## 7. Train (early stop on val RMSE)

In [ ]:
@torch.no_grad()
def eval_rmse(loader):
    model.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        pred = model(xb).cpu().numpy()
        ys.append(yb.numpy())
        ps.append(pred)
    yt = np.concatenate(ys)
    yp = np.concatenate(ps)
    return rmse(yt, yp), corr(yt, yp), yt, yp


EPOCHS = 40
PATIENCE = 8
LR = 1e-3

opt = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()

best_val = float("inf")
best_state = None
stall = 0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total, n = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        opt.step()
        total += loss.item() * len(xb)
        n += len(xb)
    train_mse = total / n

    val_rmse, val_corr, _, _ = eval_rmse(val_loader)
    history.append((epoch, train_mse, val_rmse, val_corr))
    print(
        f"epoch {epoch:02d}  train_mse={train_mse:.4f}  "
        f"val_rmse={val_rmse:.3f}  val_corr={val_corr:.3f}"
    )

    if val_rmse < best_val - 1e-4:
        best_val = val_rmse
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        stall = 0
    else:
        stall += 1
        if stall >= PATIENCE:
            print(f"early stop at epoch {epoch}")
            break

model.load_state_dict(best_state)
print("restored best val_rmse=", best_val)

## 8. Test score vs baselines

In [ ]:
test_rmse, test_corr, y_true, y_pred = eval_rmse(test_loader)
_, pers_rmse, pers_corr = baselines["persistence"]
_, clim_rmse, clim_corr = baselines["climatology"]

print("=== TEST (lead 3) ===")
print(f"model         RMSE={test_rmse:.3f}  corr={test_corr:.3f}")
print(f"persistence   RMSE={pers_rmse:.3f}  corr={pers_corr:.3f}")
print(f"climatology   RMSE={clim_rmse:.3f}  corr={clim_corr:.3f}")

if test_rmse < pers_rmse:
    print("\n✓ model beats persistence on RMSE (Day 2 win)")
else:
    print("\n○ model does not beat persistence yet — Day 3 is for tuning")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(y_true, label="true ONI", linewidth=2)
ax.plot(y_pred, label="model", alpha=0.85)
ax.plot(pers_test, label="persistence", alpha=0.6)
ax.set_title("Test set: true ONI vs lead-3 forecasts")
ax.set_xlabel("test sample index")
ax.set_ylabel("ONI")
ax.legend()
ax.axhline(0.5, color="red", ls="--", lw=0.8, alpha=0.5)
ax.axhline(-0.5, color="blue", ls="--", lw=0.8, alpha=0.5)
plt.tight_layout()
plt.show()

## 9. Save checkpoint for Day 3

In [ ]:
ckpt = {
    "model_state": best_state,
    "window": WINDOW,
    "lead": LEAD,
    "test_rmse": test_rmse,
    "test_corr": test_corr,
    "pers_rmse": pers_rmse,
    "pers_corr": pers_corr,
}
path = OUT_DIR / "enso_cnn_lead3.pt"
torch.save(ckpt, path)
print("Wrote", path)
print("Day 2 checkpoint: samples + baselines + trained lead-3 CNN.")